In [1]:
import torch
import torch.nn as nn


In [9]:
import torch
import torch.nn as nn

class MPSBatch(nn.Module):
    """
    An MPS that takes input of shape (batch_size, n, 2) and outputs (batch_size, 2).
    
    Open boundary conditions:
      A0.shape        = (1, 2, D)
      A[k].shape      = (D, 2, D) for k = 1..n-2
      A[n-1].shape    = (D, 2, 2)
    """
    def __init__(self, n, bond_dim=10):
        super().__init__()
        self.n = n
        self.D = bond_dim
        
        # A0:  (1, 2, D)
        A0 = torch.randn(1, 2, bond_dim) * 0.1
        self.A0 = nn.Parameter(A0)
        
        # A1..A_{n-2}: (D, 2, D)
        for k in range(1, n-1):
            Ak = torch.randn(bond_dim, 2, bond_dim) * 0.1
            self.register_parameter(f"A{k}", nn.Parameter(Ak))
        
        # A_{n-1}: (D, 2, 2)
        A_last = torch.randn(bond_dim, 2, 2) * 0.1
        self.A_last = nn.Parameter(A_last)

    def forward(self, x):
        """
        x: shape = (batch_size, n, 2)
        returns: shape = (batch_size, 2)
        """
        batch_size = x.shape[0]
        # We'll do an iterative contraction from left to right for each sample.
        
        # Step 0: multiply by A0
        # A0 shape = (1, 2, D)
        # x[:, 0, :] shape = (batch_size, 2)
        # We'll produce 'tmp' shape = (batch_size, D).
        
        # out_{b, d} = sum_{i} A0[0, i, d] * x[b, 0, i]
        # We'll do this with torch.einsum or a manual matmul approach.
        
        A0 = self.A0  # shape (1,2,D)
        tmp = torch.einsum('b i, i d -> b d',
                           x[:, 0, :],  # (batch, 2)
                           A0[0, :, :]) # (2, D)
        # tmp shape = (batch_size, D)
        
        # Now for k in 1..n-2:
        # A[k].shape = (D, 2, D)
        # x[:, k, :].shape = (batch_size, 2)
        # result shape = (batch_size, D)
        for k in range(1, self.n-1):
            A_k = getattr(self, f"A{k}")
            # out_{b, d_out} = sum_{d_in, i} tmp[b, d_in] * A_k[d_in, i, d_out] * x[b, k, i]
            tmp = torch.einsum('b d_in, d_in i d_out, b i -> b d_out',
                               tmp,   # (batch_size, D_in)
                               A_k,   # (D_in, 2, D_out)
                               x[:, k, :])  # (batch_size, 2)
        
        # Finally multiply by A_last: shape = (D, 2, 2)
        # That yields shape = (batch_size, 2)
        # out_{b, out2} = sum_{d_in, i} tmp[b, d_in] * A_last[d_in, i, out2] * x[b, n-1, i]
        
        tmp = torch.einsum('b d_in, d_in i out, b i -> b out',
                           tmp,          # (batch_size, D_in)
                           self.A_last,  # (D_in, 2, 2)
                           x[:, self.n-1, :])  # (batch_size, 2)
        
        # tmp shape = (batch_size, 2)
        return tmp


In [24]:
import torch
import torch.nn as nn

# ============================================================================
# MPS State Class
# ============================================================================

class MPSState(nn.Module):
    r"""An MPS state representing a wavefunction as a product of tensors.
    
    We write the MPS with open boundary conditions as:
      A₀:  shape (1, d, D)
      Aₖ:  shape (D, d, D) for 1 ≤ k ≤ n-2
      Aₙ₋₁: shape (D, d, 1)
    where d (here 2) is the local physical dimension and D is the maximal bond dimension.
    
    The forward() function expects a batch of product–states x of shape (B, n, d)
    (each local vector is assumed normalized) and returns the inner product 
    ⟨ψₘₚₛ|ψ(x)⟩ for each sample.
    """
    def __init__(self, n, bond_dim, physical_dim=2):
        super().__init__()
        self.n = n
        self.D = bond_dim  # maximal (initial) bond dimension
        self.d = physical_dim
        
        # A₀: shape (1, d, D)
        self.A0 = nn.Parameter(torch.randn(1, self.d, self.D))
        
        # Intermediate tensors A₁,...,Aₙ₋₂: each of shape (D, d, D)
        for k in range(1, n - 1):
            A = torch.randn(self.D, self.d, self.D)
            self.register_parameter(f"A{k}", nn.Parameter(A))
        
        # Final tensor Aₙ₋₁: shape (D, d, 1)
        self.A_last = nn.Parameter(torch.randn(self.D, self.d, 1))
    
    def forward(self, x):
        r"""Compute the inner product ⟨ψₘₚₛ|ψ(x)⟩ for each sample.
        
        x : Tensor of shape (B, n, d) with each local vector normalized.
        
        Returns: Tensor of shape (B,), one amplitude per sample.
        """
        B = x.shape[0]
        # (Optionally) re-normalize the local product states.
        x_norm = x / (x.norm(dim=-1, keepdim=True) + 1e-10)
        
        # --- First site ---
        # Contract x_norm[:,0,:] (B x d) with A₀[0,:,:] (d x D): result is (B x D)
        tmp = torch.einsum('Bi,ia->Ba', x_norm[:, 0, :], self.A0[0, :, :])
        
        # --- Intermediate sites ---
        for k in range(1, self.n - 1):
            A_k = getattr(self, f"A{k}")  # shape: (D, d, D)
            tmp = torch.einsum('Ba,aib,Bi->Bb', tmp, A_k, x_norm[:, k, :])
        
        # --- Last site ---
        out = torch.einsum('Ba,aic,Bi->Bc', tmp, self.A_last, x_norm[:, self.n - 1, :])
        return out.squeeze(-1)  # shape: (B,)

# ============================================================================
# MPS Norm and Normalization Functions
# ============================================================================

def mps_norm(mps):
    r"""Compute the norm of the MPS state using a transfer–matrix contraction.
    
    For an MPS with tensors A₀, A₁, …, Aₙ₋₁ the squared norm is computed 
    iteratively. (This routine assumes open boundary conditions.)
    """
    # --- Site 0 ---
    # A₀ has shape (1, d, D). We contract over the physical index.
    A0 = mps.A0[0, :, :]  # shape: (d, D)
    # L is a matrix of shape (D, D): L = A0^† A0 (summed over physical index).
    L = A0.conj().T @ A0  # shape: (D, D)
    
    # --- Sites 1 to n-1 ---
    for k in range(1, mps.n):
        if k < mps.n - 1:
            A_k = getattr(mps, f"A{k}")  # shape: (D, d, D)
        else:
            A_k = mps.A_last         # shape: (D, d, 1)
        D_left, d, D_right = A_k.shape
        L_new = torch.zeros((D_right, D_right), dtype=A_k.dtype, device=A_k.device)
        # Sum over the physical index.
        for i in range(mps.d):
            # A_k[:, i, :] has shape (D_left, D_right).
            # Update: L_new += A_k[:, i, :].conj().T @ L @ A_k[:, i, :]
            L_new += A_k[:, i, :].conj().T @ L @ A_k[:, i, :]
        L = L_new
    # For an MPS with open boundaries, L is a 1x1 matrix.
    norm2 = L.squeeze().item()
    return norm2**0.5

def normalize_mps_parameters(mps):
    r"""Normalize the MPS state by dividing each tensor by f = norm^(1/n).
    
    That is, if the MPS wavefunction is given by the product of n tensors,
    then scaling each tensor by f results in an overall scaling by fⁿ.
    To set the norm to one we choose f such that fⁿ = norm.
    """
    norm = mps_norm(mps)
    if norm == 0:
        raise ValueError("MPS norm is zero, cannot normalize.")
    factor = norm ** (1.0 / mps.n)
    with torch.no_grad():
        mps.A0.data.div_(factor)
        for k in range(1, mps.n - 1):
            getattr(mps, f"A{k}").data.div_(factor)
        mps.A_last.data.div_(factor)

# ============================================================================
# Training the MPS State
# ============================================================================

def train_mps(mps, num_epochs=200, batch_size=32):
    r"""Train the MPS so that for a batch of normalized product–states x,
    the inner product ⟨ψₘₚₛ|ψ(x)⟩ is as close as possible to 1.
    
    (Here the target is taken to be 1 for every sample; adjust as needed.)
    """
    optimizer = torch.optim.Adam(mps.parameters(), lr=1e-4)
    
    for epoch in range(num_epochs):
        # Generate a batch of product–states.
        # For example, sample angles uniformly and form (cosθ, sinθ).
        x = torch.randn(batch_size, mps.n, 2)
        # Ensure the last element has an even hamming weight
        for i in range(batch_size):
            hamming_weight = (x[i, -1, :] > 0).sum().item()
            if hamming_weight % 2 != 0:
                x[i, -1, 0] = -x[i, -1, 0]
        psi = torch.stack([torch.cos(x), torch.sin(x)], dim=-1)  # shape: (B, n, 2)
        
        # Target inner product is 1.
        target = torch.ones(batch_size)
        
        inner_prod = mps(x)
        loss = ((inner_prod - target) ** 2).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # After each update, re-normalize the MPS.
        normalize_mps_parameters(mps)
        
        if epoch % 20 == 0 or epoch == 0:
            current_norm = mps_norm(mps)
            print(f"Epoch {epoch:03d}  Loss: {loss.item():.6f}   MPS norm: {current_norm:.6f}")

# ============================================================================
# Example Usage
# ============================================================================

n_sites = 10    # number of sites
bond_dim = 20   # initial maximal bond dimension
mps_state = MPSState(n_sites, bond_dim)

# Normalize the initial MPS.
normalize_mps_parameters(mps_state)
print("Initial MPS norm:", mps_norm(mps_state))

# Train the MPS so that for each normalized product state, ⟨ψₘₚₛ|ψ(x)⟩ is close to 1.
train_mps(mps_state, num_epochs=4000, batch_size=32)

# Final check.
normalize_mps_parameters(mps_state)
print("\nFinal MPS norm:", mps_norm(mps_state))
print("Sample inner products on a random batch:")
angles = 2 * 3.14159 * torch.rand(5, n_sites)
x_test = torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)
inner_products = mps_state(x_test)
print(inner_products)


Initial MPS norm: 1.0000003576278047
Epoch 000  Loss: 1.010418   MPS norm: 1.000000
Epoch 020  Loss: 1.009340   MPS norm: 1.000000
Epoch 040  Loss: 0.982744   MPS norm: 1.000000
Epoch 060  Loss: 1.006169   MPS norm: 1.000000
Epoch 080  Loss: 1.017246   MPS norm: 1.000000
Epoch 100  Loss: 0.998778   MPS norm: 1.000000
Epoch 120  Loss: 1.007745   MPS norm: 1.000000
Epoch 140  Loss: 1.006892   MPS norm: 1.000000
Epoch 160  Loss: 0.997192   MPS norm: 1.000000
Epoch 180  Loss: 1.002628   MPS norm: 1.000000
Epoch 200  Loss: 1.013752   MPS norm: 1.000000
Epoch 220  Loss: 0.985627   MPS norm: 1.000000
Epoch 240  Loss: 1.007170   MPS norm: 1.000000
Epoch 260  Loss: 1.006504   MPS norm: 1.000000
Epoch 280  Loss: 1.004533   MPS norm: 1.000000
Epoch 300  Loss: 1.010825   MPS norm: 1.000000
Epoch 320  Loss: 0.993600   MPS norm: 1.000000
Epoch 340  Loss: 1.013289   MPS norm: 1.000000
Epoch 360  Loss: 0.997669   MPS norm: 1.000000
Epoch 380  Loss: 1.001880   MPS norm: 1.000000
Epoch 400  Loss: 1.0071